## NBA 3.1 | PX Patient Features

Builds First Occurrence, Recency, Frequency, and Hospitalization features for PX claims.

**Input files:**
- All features (First Occ, Recency, Frequency): `px_all` (673M rows, 2021-01-02→2026-08-31)
  - Note: `px_full_history` was removed from INPUT_PATHS_31 — it was an identical duplicate of `px_all`. Use `px_all` only.
- Hospitalization: `px_hosp` (pre-filtered, 315M rows, 2021-01-02→2026-08-31)

**Output:** `.../kerendia_nba_3_1_volume/notebook_output/patient_features/px/`

**Skill:** `databricks-skills/patient_features_skills/SKILL.md` (Sections 3–6 incl. NBA 3.1 removal notes). Builders come from `00_Patient_Feature_Helpers_31` (optimized, validated). Section 13 add-on features are built in `01_Additional_Patient_Features`.

**NBA 3.1 changes applied:** censored first-occurrence events dropped / capped at 365 d + `_FIRST_CLAIM_BEFORE_WINDOW` flag; unsuffixed recency totals never written; `LOGRATIO_FREQ_*` (N = 3, 6); unique patients for 90/180/360 d only; `FREQ_*_IN_LAST_*` dropped where `NUM_OF_PATIENTS_*` exists. Frequency and hospitalization outputs = one row per HCP x cohort at `SVC_DT_M = LOOKBACK_END_DATE`.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/02_Input_Data_Creation/00_event_list_config"

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/00_Helper_Notebooks/00_cohort_generation"

In [0]:
%run "/Workspace/Users/sehar.aejaz.ext@bayer.com/kerendia_nba_3_1_develop/NBA 3.1 - Dev/03_patient_features/Final Notebooks/00_Patient_Feature_Helpers_31"

In [0]:
import re
from functools import reduce
from pyspark.sql import functions as F, Window

FEATURE_FAMILY  = "claims"
CLAIM_TYPE      = "px"
EVENT_TYPE_VAL  = "px"

# PX: single non-hosp file (no windowed variant exists)
PX_ALL_PATH  = INPUT_PATHS_31["px_all"]   # First Occ + Recency + Frequency
PX_HOSP_PATH = INPUT_PATHS_31["px_hosp"]  # Hospitalization (pre-filtered)

OUTPUT_BASE  = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
                "kerendia_nba_3/kerendia_nba_3_1_volume/notebook_output/patient_features/px")
FO_OUT   = f"{OUTPUT_BASE}/px_first_occurrence_features"
REC_OUT  = f"{OUTPUT_BASE}/px_recency_features"
FREQ_OUT = f"{OUTPUT_BASE}/px_frequency_features"
HOSP_OUT = f"{OUTPUT_BASE}/px_hospitalization_features"

dbutils.widgets.text("TEST_HCP_SAMPLE", "0", "HCP sample size (0=full)")
TEST_HCP_SAMPLE  = int(dbutils.widgets.get("TEST_HCP_SAMPLE"))
if TEST_HCP_SAMPLE > 0:   # sample runs write to their own folder so full-universe outputs are never overwritten
    OUTPUT_BASE = f"{OUTPUT_BASE}/_test_sample_{TEST_HCP_SAMPLE}"
    FO_OUT, REC_OUT, FREQ_OUT, HOSP_OUT = [f"{OUTPUT_BASE}/{p.rsplit('/', 1)[1]}" for p in (FO_OUT, REC_OUT, FREQ_OUT, HOSP_OUT)]
RECENCY_WINDOWS  = [30, 60, 90, 180, 270, 360]
LAG_WINDOWS      = [1, 2, 3, 6, 12]
CHANGE_WINDOWS   = [3, 6]
PATIENT_WINDOWS  = [90, 180, 360]          # NBA 3.1 removal note: 90/180/360 only
DROP_FREQ_WITH_PATIENT_EQUIV = True        # NBA 3.1: drop FREQ_*_IN_LAST_* when a NUM_OF_PATIENTS_* equivalent exists
TMP_BASE         = f"{OUTPUT_BASE}/_tmp"   # Delta checkpoints for intermediates (Section 7.4), removed at the end
HOSP_LAG_WINDOWS = [1, 6, 12]
HOSP_PAT_WINDOWS = [30, 180, 360]
EVENT_COLLAPSED  = "ALL_PROCEDURES"

print(f"PX_ALL_PATH  : {PX_ALL_PATH}")
print(f"PX_HOSP_PATH : {PX_HOSP_PATH}")
print(f"OUTPUT_BASE  : {OUTPUT_BASE}")

In [0]:
cohorts = CohortGenerator(FEATURE_FAMILY).load()
meta = cohorts.attrs["metadata"]
assert FEATURE_FAMILY == "claims" and meta["feature_family"] == "claims" and meta["offset_days"] == 0
assert list(cohorts["cohort_id"]) == [f"c{i}" for i in range(1, 29)]
c1 = cohorts.iloc[0]
assert (str(c1["lookback_start_date"]), str(c1["lookback_end_date"])) == ("2023-06-01", "2024-05-01")
print(f"Cohorts loaded: {len(cohorts)} rows")

In [0]:
HCP_UNIVERSE_PATH = ("/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/"
                     "kerendia_nba_3/kerendia_nba_3_1_volume/HCP_Universe_without_call_hcps")
hcp_universe_df = spark.read.parquet(HCP_UNIVERSE_PATH).select("PROVIDER_ID").distinct()
if TEST_HCP_SAMPLE > 0:
    hcp_universe_df = (hcp_universe_df.orderBy(F.xxhash64("PROVIDER_ID"), "PROVIDER_ID")
                                      .limit(TEST_HCP_SAMPLE))
hcp_universe = F.broadcast(hcp_universe_df)
px_event_keys = event_groups("px")
print(f"Universe HCPs: {'TEST:'+str(TEST_HCP_SAMPLE) if TEST_HCP_SAMPLE > 0 else '294,727 (full, without call HCPs)'}")
print(f"Px EVENT_GROUPs: {len(px_event_keys)}")

In [0]:
claims_all = (spark.read.format("delta").load(PX_ALL_PATH)
               .join(hcp_universe, "PROVIDER_ID", "left_semi"))
claims_all = standardize(claims_all, EVENT_TYPE_VAL)
validate_event_input(claims_all, CLAIM_TYPE)
claims_freq = (claims_all
               .withColumn("SVC_DT_M", F.trunc(F.col("EVENT_DATE"), "month"))
               .select("PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "SVC_DT_M",
                       "EVENT_DATE", "CLAIM_ID"))
print("px_all loaded and standardized")

In [0]:
# Pre-aggregate -> pivot -> cross-join cohorts, plus the NBA 3.1 censoring rule (Section 3 removal note):
#   >50% of HCPs first seen within 60 d of the data start -> dropped; <=10% -> raw; else min(days, 365) + flag
fo, fo_censor_report = build_first_occurrence(claims_all, px_event_keys, cohorts, tmp_path=f"{TMP_BASE}/_fo_pre")
print(fo_censor_report["RULE"].value_counts().to_string())
print("Dropped (left-censored):", fo_censor_report.loc[fo_censor_report["RULE"] == "DROP_CENSORED", "EVENT_SAN"].tolist())
(spark.createDataFrame(fo_censor_report.astype({"DATA_START": str}))
      .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
      .save(f"{OUTPUT_BASE}/px_first_occurrence_censor_report"))
(fo.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .partitionBy("COHORT").save(FO_OUT))
print(f"Written: {FO_OUT}")

In [0]:
# px_all is full history: the builder restricts to the union of the 28 lookback windows first.
recency = build_recency(claims_all, px_event_keys, cohorts, suffix="PX", windows=RECENCY_WINDOWS,
                        count_claim_ids=True, tmp_base=f"{TMP_BASE}/rec")
(recency.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .partitionBy("COHORT").save(REC_OUT))
print(f"Written: {REC_OUT}")

In [0]:
# Optimized Frequency (Section 5): long-format monthly counts, rolling sums only at the 28 cohort snapshot
# months in ONE conditional aggregation, LOGRATIO_FREQ_* (N = 3, 6), unique patients 90/180/360 d via a
# per-patient month bitmask (one pass instead of 28 x 5 scans). Output: one row per HCP x cohort.
frequency = build_frequency(claims_freq, px_event_keys, cohorts, patient_prefix="DID_PROCEDURE",
                            lag_windows=LAG_WINDOWS, change_windows=CHANGE_WINDOWS,
                            patient_windows=PATIENT_WINDOWS,
                            drop_freq_with_patient_equiv=DROP_FREQ_WITH_PATIENT_EQUIV,
                            tmp_base=f"{TMP_BASE}/freq")
(frequency.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
          .partitionBy("COHORT").save(FREQ_OUT))
print(f"Written: {FREQ_OUT} | columns: {len(frequency.columns)}")

In [0]:
# Pre-filtered PX hosp file (no DATA_SOURCE filter needed)
hosp_claims = (spark.read.format("delta").load(PX_HOSP_PATH)
                .join(hcp_universe, "PROVIDER_ID", "left_semi"))
hosp_claims = (standardize(hosp_claims, EVENT_TYPE_VAL)
               .withColumn("SVC_DT_M", F.trunc(F.col("EVENT_DATE"), "month"))
               .select("PROVIDER_ID", "PATIENT_ID", "SVC_DT_M"))

hosp_final = build_hospitalization(claims_freq.select("PROVIDER_ID", "PATIENT_ID", "SVC_DT_M"), hosp_claims,
                                   cohorts, collapsed=EVENT_COLLAPSED, patient_label="DID_PROCEDURE",
                                   lag_windows=HOSP_LAG_WINDOWS, patient_windows=HOSP_PAT_WINDOWS)
(hosp_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
           .partitionBy("COHORT").save(HOSP_OUT))
print(f"Written: {HOSP_OUT}")

In [0]:
for path, label in [(FO_OUT, "First Occurrence"), (REC_OUT, "Recency"),
                    (FREQ_OUT, "Frequency"), (HOSP_OUT, "Hospitalization")]:
    df = spark.read.format("delta").load(path)
    cohort_ids = {r[0] for r in df.select("COHORT").distinct().collect()}
    expected   = set(cohorts["cohort_id"])
    n_outside  = (df.select("PROVIDER_ID").distinct()
                    .join(hcp_universe_df, "PROVIDER_ID", "left_anti").count())
    print(f"{label}: {df.count():,} rows | cohorts={len(cohort_ids)}/28 "
          f"| missing={expected-cohort_ids or 'none'} | outside universe={n_outside}")
dbutils.fs.rm(TMP_BASE, True)   # drop intermediate checkpoints
dbutils.notebook.exit("PX patient features written: First Occurrence, Recency, Frequency, Hospitalization")